In [1]:
!pip install -q -U --no-cache-dir \
  git+https://github.com/huggingface/transformers.git \
  "mistral-common>=1.8.6" \
  accelerate \
  pandas \
  scikit-learn \
  tqdm \
  huggingface_hub

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 169.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 204.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 222.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 200.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 194.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 227.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the followi

In [2]:
from google.colab import files

uploaded = files.upload()

Saving audio_files.zip to audio_files.zip


In [5]:
import os
import re
import pandas as pd
import torch

from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from transformers import Mistral3ForConditionalGeneration, MistralCommonBackend

file_path = "test_dataset_final.csv"

df = pd.read_csv(file_path)

print("CSV preview:")
print(df.head())
print("Columns:", df.columns)

text_col = "transcript"
label_col = "label"

df = df.dropna(subset=[text_col, label_col]).reset_index(drop=True)

labels = sorted(df[label_col].astype(str).unique().tolist())

print("Text column:", text_col)
print("Label column:", label_col)
print("Labels:", labels)

model_name = "mistralai/Ministral-3-3B-Instruct-2512"

print("\nLoading model:", model_name)

if not torch.cuda.is_available():
    raise RuntimeError("Please enable GPU in Colab: Runtime -> Change runtime type -> GPU")

tokenizer = MistralCommonBackend.from_pretrained(model_name)

model = Mistral3ForConditionalGeneration.from_pretrained(
    model_name,
    device_map="auto"
).eval()

model.tie_weights()

print("Model loaded successfully.")

def make_prompt(sent):
    prompt = f"""You are a sentence type classifier.

Classify the sentence into exactly one of these labels:

declarative: a normal statement or information-giving sentence.
interrogative: a question or sentence asking for information.
imperative: a command, request, advice, instruction, or order.
exclamatory: a sentence showing strong emotion, surprise, excitement, or emphasis.

Examples:
Sentence: I am going to school today.
Label: declarative

Sentence: What are you doing here?
Label: interrogative

Sentence: Please close the door.
Label: imperative

Sentence: What a beautiful day!
Label: exclamatory

Sentence: Give me scotch, please.
Label: imperative

Sentence: I heard you and James are engaged at last.
Label: declarative

Sentence: Hi, what brings you to my office today?
Label: interrogative

Now classify this sentence.

Sentence:
{sent}

Return only one label from:
declarative, exclamatory, imperative, interrogative

Answer with only the label.

Label:"""

    return prompt

def clean_pred(ans):
    raw = ans
    ans = ans.strip().lower()

    ans = ans.replace("answer:", " ")
    ans = ans.replace("label:", " ")

    ans = re.sub(r"[^a-zA-Z]", " ", ans)
    ans = " ".join(ans.split())

    words = ans.split()

    for lab in labels:
        if ans == lab.lower():
            return lab

    for w in words:
        for lab in labels:
            if w == lab.lower():
                return lab

    for lab in labels:
        if lab.lower() in ans:
            return lab

    print("Could not clean prediction. Raw answer was:", raw)
    return ans

def predict(sent, i=None, debug=False):
    prompt = make_prompt(sent)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True
    )

    input_len = inputs["input_ids"].shape[-1]

    inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )[0]

    new_tokens = outputs[input_len:]

    new_text = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    full_text = tokenizer.decode(
        outputs,
        skip_special_tokens=True
    ).strip()

    if debug:
        print("\n" + "=" * 80)
        print("DEBUG SAMPLE:", i)
        print("Input sentence:", sent)
        print("Raw decoded FULL text:")
        print(full_text)
        print("\nRaw decoded NEW text only:")
        print(new_text)
        print("=" * 80)

    if len(new_text.strip()) > 0:
        ans = new_text
    else:
        ans = full_text.split("Label:")[-1]

    return clean_pred(ans)

print("\nRunning debug predictions on first 5 samples:")

debug_preds = []

for i in range(min(5, len(df))):
    sent = str(df.loc[i, text_col])
    true_label = str(df.loc[i, label_col])
    pred = predict(sent, i=i, debug=True)
    debug_preds.append(pred)

    print("Sample:", i)
    print("Sentence:", sent)
    print("True label:", true_label)
    print("Predicted label:", pred)

print("\nDebug first 5 predictions:", debug_preds)

print("\nNow running on full dataset...")

preds = []

for sent in tqdm(df[text_col].astype(str).tolist()):
    preds.append(predict(sent))

df["predicted_type"] = preds

print("\nPrediction counts:")
print(df["predicted_type"].value_counts())

print("\nFirst 20 predictions:")
print(df[[text_col, label_col, "predicted_type"]].head(20))

y_true = df[label_col].astype(str).tolist()
y_pred = df["predicted_type"].astype(str).tolist()

acc = accuracy_score(y_true, y_pred)

p, r, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy:", acc)
print("Precision:", p)
print("Recall:", r)
print("F1 Score:", f1)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, zero_division=0))

df.to_csv("ministral3_text_predictions.csv", index=False)

print("Saved predictions to ministral3_text_predictions.csv")

from google.colab import files
files.download("ministral3_text_predictions.csv")

CSV preview:
      file_name                                         transcript  \
0   0_0_d45.wav     Hello, Jane. I'm really glad to meet you here.   
1   0_0_d47.wav         I heard you and James are engaged at last.   
2  0_0_d215.wav                            Give me scotch, please.   
3   0_0_d49.wav            Hi, what brings you to my office today?   
4  0_0_d159.wav  Please help yourself at your dishes. I hope yo...   

           label  
0    declarative  
1    declarative  
2     imperative  
3  interrogative  
4     imperative  
Columns: Index(['file_name', 'transcript', 'label'], dtype='str')
Text column: transcript
Label column: label
Labels: ['declarative', 'exclamatory', 'imperative', 'interrogative']

Loading model: mistralai/Ministral-3-3B-Instruct-2512


tekken.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.90k [00:00<?, ?B/s]

[transformers] FP8 quantized models is only supported on GPUs with compute capability >= 8.9 (e.g 4090/H100), actual = `7.5`. We will default to dequantizing the model to bf16. Feel free to use a different quantization method like bitsandbytes or torchao


model.safetensors:   0%|          | 0.00/4.67G [00:00<?, ?B/s]

...


DEBUG SAMPLE: 3
Input sentence: Hi, what brings you to my office today?
Raw decoded FULL text:
You are a sentence type classifier.

Classify the sentence into exactly one of these labels:

declarative: a normal statement or information-giving sentence.
interrogative: a question or sentence asking for information.
imperative: a command, request, advice, instruction, or order.
exclamatory: a sentence showing strong emotion, surprise, excitement, or emphasis.

Examples:
Sentence: I am going to school today.
Label: declarative

Sentence: What are you doing here?
Label: interrogative

Sentence: Please close the door.
Label: imperative

Sentence: What a beautiful day!
Label: exclamatory

Sentence: Give me scotch, please.
Label: imperative

Sentence: I heard you and James are engaged at last.
Label: declarative

Sentence: Hi, what brings you to my office today?
Label: interrogative

Now classify this sentence.

Sentence:
Hi, what brings you to my office today?

Return only one label from:
de

100%|██████████| 404/404 [05:28<00:00,  1.23it/s]


Prediction counts:
predicted_type
imperative       167
interrogative    101
exclamatory       99
declarative       37
Name: count, dtype: int64

First 20 predictions:
                                           transcript          label  \
0      Hello, Jane. I'm really glad to meet you here.    declarative   
1          I heard you and James are engaged at last.    declarative   
2                             Give me scotch, please.     imperative   
3             Hi, what brings you to my office today?  interrogative   
4   Please help yourself at your dishes. I hope yo...     imperative   
5                                   Hello, Jack here.    declarative   
6             Excuse me. I wonder if you can help me.     imperative   
7             Umm, Jenny, are you having a good time?  interrogative   
8   Excuse me. I bought this shirt yesterday, but ...     imperative   
9      Watch out! You're too close to the fire place.    exclamatory   
10                          Come on! It'

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
!pip uninstall -y transformers tokenizers mistral-common -q

!pip install -q -U --no-cache-dir \
  git+https://github.com/huggingface/transformers.git \
  "mistral-common[audio]>=1.8.6" \
  accelerate \
  librosa \
  soundfile \
  pandas \
  scikit-learn \
  tqdm \
  huggingface_hub

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 358.1 MB/s eta 0:00:00


In [4]:
from google.colab import files

uploaded = files.upload()
import zipfile
import os
import glob

zip_files = glob.glob("audio_files*.zip")
print("Found zip files:", zip_files)

zip_path = zip_files[0]
audio_dir = "audio_files"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(".")

print("Files extracted.")
print("Checking audio folder:")
print(os.listdir(audio_dir)[:10])
print("Total audio files:", len(os.listdir(audio_dir)))

Saving test_dataset_final.csv to test_dataset_final.csv
Found zip files: ['audio_files.zip']
Files extracted.
Checking audio folder:
['1_1_d2228.wav', '0_1_d1479.wav', '13_0_d219.wav', '1_1_d1946.wav', '1_1_d1447.wav', '0_1_d2036.wav', '0_1_d1877.wav', '1_1_d2342.wav', '0_0_d47.wav', '12_0_d1177.wav']
Total audio files: 404


In [7]:
import os
import re
import pandas as pd
import torch

from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from transformers import AutoProcessor, VoxtralForConditionalGeneration

# free whatever colab already allocated before we touch the model
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

file_path = "test_dataset_final.csv"
audio_dir = "audio_files"

df = pd.read_csv(file_path)

print("CSV preview:")
print(df.head())
print("Columns:", df.columns)

text_col = "transcript"
audio_col = "file_name"
label_col = "label"

df = df.dropna(subset=[text_col, audio_col, label_col]).reset_index(drop=True)

labels = sorted(df[label_col].astype(str).unique().tolist())

print("Audio column:", audio_col)
print("Text column:", text_col)
print("Label column:", label_col)
print("Labels:", labels)

missing = []

for f in df[audio_col].astype(str).tolist():
    p = os.path.join(audio_dir, f)
    if not os.path.exists(p):
        missing.append(f)

print("Missing audio files:", len(missing))

if len(missing) > 0:
    print("First missing files:", missing[:10])
    raise FileNotFoundError("Some audio files from CSV are missing from audio_files folder.")

print("\nChecking first 3 audio-text pairs:")

for i in range(min(3, len(df))):
    audio_path = os.path.join(audio_dir, str(df.loc[i, audio_col]))
    transcript = str(df.loc[i, text_col])
    label = str(df.loc[i, label_col])

    print("\nSample", i)
    print("Audio path:", audio_path)
    print("Audio exists:", os.path.exists(audio_path))
    print("Audio size bytes:", os.path.getsize(audio_path))
    print("Transcript:", transcript)
    print("True label:", label)

model_name = "mistralai/Voxtral-Mini-3B-2507"

print("\nLoading model:", model_name)

if not torch.cuda.is_available():
    raise RuntimeError("Please enable GPU in Colab: Runtime -> Change runtime type -> GPU")

device = "cuda"
dtype = torch.float16

processor = AutoProcessor.from_pretrained(model_name)

tok = processor.tokenizer

print("\nBefore fixing pad token:")
print("pad_token:", tok.pad_token)
print("pad_token_id:", tok.pad_token_id)
print("eos_token:", tok.eos_token)
print("eos_token_id:", tok.eos_token_id)

if tok.pad_token is None or tok.pad_token_id is None or tok.pad_token_id < 0:
    tok.pad_token = tok.eos_token
    tok.pad_token_id = tok.eos_token_id

tok.padding_side = "left"

print("\nAfter fixing pad token:")
print("pad_token:", tok.pad_token)
print("pad_token_id:", tok.pad_token_id)
print("padding_side:", tok.padding_side)

model = VoxtralForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map=device
).eval()

model.config.pad_token_id = processor.tokenizer.pad_token_id
model.generation_config.pad_token_id = processor.tokenizer.pad_token_id

print("Model loaded successfully.")

def make_prompt(transcript):
    prompt = f"""You are a sentence type classifier.

You will receive both the speech audio and its transcript.

Classify the sentence into exactly one of these labels:

declarative: a normal statement or information-giving sentence.
interrogative: a question or sentence asking for information.
imperative: a command, request, advice, instruction, or order.
exclamatory: a sentence showing strong emotion, surprise, excitement, or emphasis.

Use both the audio and the transcript. The transcript gives the words, and the audio may give tone, emphasis, and intonation.

Examples:
Sentence: I am going to school today.
Label: declarative

Sentence: What are you doing here?
Label: interrogative

Sentence: Please close the door.
Label: imperative

Sentence: What a beautiful day!
Label: exclamatory

Sentence: Give me scotch, please.
Label: imperative

Sentence: I heard you and James are engaged at last.
Label: declarative

Sentence: Hi, what brings you to my office today?
Label: interrogative

Now classify this sentence.

Transcript:
{transcript}

Return only one label from:
declarative, exclamatory, imperative, interrogative

Answer with only the label.

Label:"""

    return prompt

def clean_pred(ans):
    raw = ans
    ans = ans.strip().lower()

    ans = ans.replace("answer:", " ")
    ans = ans.replace("label:", " ")

    ans = re.sub(r"[^a-zA-Z]", " ", ans)
    ans = " ".join(ans.split())

    words = ans.split()

    for lab in labels:
        if ans == lab.lower():
            return lab

    for w in words:
        for lab in labels:
            if w == lab.lower():
                return lab

    for lab in labels:
        if lab.lower() in ans:
            return lab

    print("Could not clean prediction. Raw answer was:", raw)
    return ans

def print_input_debug(i, audio_path, transcript, conversation, inputs):
    print("\n" + "=" * 80)
    print("DEBUG SAMPLE:", i)
    print("Audio path:", audio_path)
    print("Audio exists:", os.path.exists(audio_path))
    print("Audio size bytes:", os.path.getsize(audio_path))
    print("Transcript:", transcript)

    print("\nConversation content types:")
    for msg in conversation:
        print("Role:", msg["role"])
        for item in msg["content"]:
            print("  type:", item["type"])

    print("\nProcessor input keys:")
    print(list(inputs.keys()))

    print("\nProcessor input tensor details:")
    for k, v in inputs.items():
        if torch.is_tensor(v):
            print(k, "shape:", tuple(v.shape), "dtype:", v.dtype, "device before move:", v.device)
        else:
            print(k, "type:", type(v))

    if "input_ids" in inputs:
        print("Text input token length:", inputs["input_ids"].shape[-1])

    audio_keys = [k for k in inputs.keys() if "audio" in k.lower() or "feature" in k.lower() or "input_features" in k.lower()]
    print("Audio-related keys:", audio_keys)

    print("=" * 80 + "\n")

# FIX 1: move tensors individually since dict has no .to()
def move_inputs_to_device(inputs):
    moved = {}
    for k, v in inputs.items():
        if torch.is_tensor(v):
            if v.dtype in (torch.float32, torch.float16, torch.bfloat16):
                moved[k] = v.to(device=device, dtype=dtype)
            else:
                moved[k] = v.to(device=device)
        else:
            moved[k] = v
    return moved

def predict(audio_path, transcript, i=None, debug=False):
    prompt = make_prompt(transcript)

    conversation = [
        {
            "role": "user",
            "content": [
                {
                    "type": "audio",
                    "path": audio_path
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ],
        }
    ]

    # FIX 2: must pass return_tensors and tokenize=True to get a proper dict back
    inputs = processor.apply_chat_template(
        conversation,
        tokenize=True,
        return_tensors="pt"
    )

    if debug:
        print_input_debug(i, audio_path, transcript, conversation, inputs)

    # FIX 3: also check for "input_features" which is Voxtral's audio key
    audio_keys = [
        k for k in inputs.keys()
        if "audio" in k.lower() or "feature" in k.lower() or k == "input_features"
    ]

    if len(audio_keys) == 0:
        raise ValueError(
            "Audio was not processed. Processor output has no audio/input feature keys. "
            "This usually means transformers or mistral-common[audio] is not installed correctly."
        )

    input_len = inputs["input_ids"].shape[-1]

    # FIX 4: use fixed move_inputs_to_device (dict has no .to())
    inputs = move_inputs_to_device(inputs)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=processor.tokenizer.pad_token_id
        )

    new_text = processor.batch_decode(
        outputs[:, input_len:],
        skip_special_tokens=True
    )[0].strip()

    full_text = processor.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0].strip()

    if debug:
        print("\nRaw decoded FULL text:")
        print(full_text)

        print("\nRaw decoded NEW text only:")
        print(new_text)

    if len(new_text.strip()) > 0:
        ans = new_text
    else:
        ans = full_text.split("Label:")[-1]

    pred = clean_pred(ans)

    if debug:
        print("Cleaned prediction:", pred)
        print("-" * 80)

    return pred

print("\nRunning debug predictions on first 5 samples only:")

debug_preds = []

for i in range(min(5, len(df))):
    audio_path = os.path.join(audio_dir, str(df.loc[i, audio_col]))
    transcript = str(df.loc[i, text_col])
    true_label = str(df.loc[i, label_col])

    pred = predict(audio_path, transcript, i=i, debug=True)

    debug_preds.append(pred)

    print("Sample:", i)
    print("True label:", true_label)
    print("Predicted label:", pred)

print("\nDebug first 5 predictions:", debug_preds)

print("\nNow running on full dataset...")

preds = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    audio_path = os.path.join(audio_dir, str(row[audio_col]))
    transcript = str(row[text_col])

    pred = predict(audio_path, transcript, i=i, debug=False)
    preds.append(pred)

df["predicted_type"] = preds

print("\nPrediction counts:")
print(df["predicted_type"].value_counts())

print("\nFirst 20 predictions:")
print(df[[audio_col, text_col, label_col, "predicted_type"]].head(20))

y_true = df[label_col].astype(str).tolist()
y_pred = df["predicted_type"].astype(str).tolist()

acc = accuracy_score(y_true, y_pred)

p, r, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy:", acc)
print("Precision:", p)
print("Recall:", r)
print("F1 Score:", f1)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, zero_division=0))

df.to_csv("voxtral_speech_text_predictions.csv", index=False)

print("Saved predictions to voxtral_speech_text_predictions.csv")

from google.colab import files
files.download("voxtral_speech_text_predictions.csv")

CSV preview:
      file_name                                         transcript  \
0   0_0_d45.wav     Hello, Jane. I'm really glad to meet you here.   
1   0_0_d47.wav         I heard you and James are engaged at last.   
2  0_0_d215.wav                            Give me scotch, please.   
3   0_0_d49.wav            Hi, what brings you to my office today?   
4  0_0_d159.wav  Please help yourself at your dishes. I hope yo...   

           label  
0    declarative  
1    declarative  
2     imperative  
3  interrogative  
4     imperative  
Columns: Index(['file_name', 'transcript', 'label'], dtype='str')
Audio column: file_name
Text column: transcript
Label column: label
Labels: ['declarative', 'exclamatory', 'imperative', 'interrogative']
Missing audio files: 0

Checking first 3 audio-text pairs:

Sample 0
Audio path: audio_files/0_0_d45.wav
Audio exists: True
Audio size bytes: 242594
Transcript: Hello, Jane. I'm really glad to meet you here.
True label: declarative

Sample 1
Audio 

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 8.69 GiB. GPU 0 has a total capacity of 14.56 GiB of which 7.21 GiB is free. Including non-PyTorch memory, this process has 7.35 GiB memory in use. Of the allocated memory 7.21 GiB is allocated by PyTorch, and 21.41 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [8]:
# Ministral text-only tensor / embedding analysis

import os
import torch
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files

file_path = "test_dataset_final.csv"

text_col = "transcript"
label_col = "label"

df = pd.read_csv(file_path)
df = df.dropna(subset=[text_col, label_col]).reset_index(drop=True)

N_ANALYSIS = 50
small_df = df.head(N_ANALYSIS).copy()

print("Running Ministral text-only tensor / embedding analysis")
print("Number of samples:", len(small_df))

if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError("model and tokenizer are not loaded. Run your Ministral model loading cell first.")

def make_ministral_text_prompt(sent):
    prompt = f"""You are a sentence type classifier.

Classify the sentence into exactly one of these labels:

declarative: a normal statement or information-giving sentence.
interrogative: a question or sentence asking for information.
imperative: a command, request, advice, instruction, or order.
exclamatory: a sentence showing strong emotion, surprise, excitement, or emphasis.

Examples:
Sentence: I am going to school today.
Label: declarative

Sentence: What are you doing here?
Label: interrogative

Sentence: Please close the door.
Label: imperative

Sentence: What a beautiful day!
Label: exclamatory

Sentence: Give me scotch, please.
Label: imperative

Sentence: I heard you and James are engaged at last.
Label: declarative

Sentence: Hi, what brings you to my office today?
Label: interrogative

Now classify this sentence.

Sentence:
{sent}

Return only one label from:
declarative, exclamatory, imperative, interrogative

Answer with only the label.

Label:"""

    return prompt

def build_ministral_text_inputs(sent):
    prompt = make_ministral_text_prompt(sent)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True
    )

    return inputs, prompt

def find_text_embedding_layer(model):
    candidates = []

    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Embedding):
            candidates.append((name, module))

    print("\nFound embedding layers:")

    for name, module in candidates:
        print(name, "weight shape:", tuple(module.weight.shape), "device:", module.weight.device)

    if len(candidates) == 0:
        raise ValueError("No torch.nn.Embedding layer found inside the model.")

    preferred_words = [
        "embed_tokens",
        "tok_embeddings",
        "word_embeddings",
        "wte"
    ]

    for word in preferred_words:
        for name, module in candidates:
            if word in name.lower():
                print("\nUsing text embedding layer:", name)
                return module

    candidates = sorted(
        candidates,
        key=lambda x: x[1].weight.shape[0],
        reverse=True
    )

    print("\nUsing fallback text embedding layer:", candidates[0][0])
    return candidates[0][1]

text_embedding_layer = find_text_embedding_layer(model)

def get_text_mean_embedding(inputs):
    emb_layer = text_embedding_layer
    emb_device = emb_layer.weight.device

    input_ids = inputs["input_ids"].to(emb_device)

    with torch.no_grad():
        text_emb = emb_layer(input_ids)

    text_emb = text_emb.detach().float().cpu()

    # shape: batch, token_length, hidden_dim
    mean_emb = text_emb.mean(dim=1).squeeze(0)

    return mean_emb.numpy(), tuple(text_emb.shape)

analysis_rows = []
text_vecs = []

for i, row in tqdm(small_df.iterrows(), total=len(small_df)):
    sent = str(row[text_col])
    label = str(row[label_col])

    inputs, prompt = build_ministral_text_inputs(sent)

    text_vec, text_shape = get_text_mean_embedding(inputs)

    text_vecs.append(text_vec)

    analysis_rows.append({
        "sample_id": i,
        "label": label,
        "sentence": sent,
        "input_ids_len": int(inputs["input_ids"].shape[-1]),
        "text_embedding_shape": str(text_shape),
        "text_embedding_norm": float(np.linalg.norm(text_vec)),
        "prompt_char_length": len(prompt)
    })

analysis_df = pd.DataFrame(analysis_rows)

print("\nMinistral text-only tensor / embedding analysis:")
print(analysis_df)

print("\nSummary:")
print("Average input_ids length:")
print(analysis_df["input_ids_len"].mean())

print("\nAverage text embedding norm:")
print(analysis_df["text_embedding_norm"].mean())

analysis_df.to_csv("ministral_text_embedding_analysis_n50.csv", index=False)

np.save(
    "ministral_text_mean_embeddings_n50.npy",
    np.stack(text_vecs)
)

print("\nSaved files:")
print("ministral_text_embedding_analysis_n50.csv")
print("ministral_text_mean_embeddings_n50.npy")

files.download("ministral_text_embedding_analysis_n50.csv")

Running Ministral text-only tensor / embedding analysis
Number of samples: 50

Found embedding layers:
model.language_model.embed_tokens weight shape: (131072, 3072) device: cuda:0

Using text embedding layer: model.language_model.embed_tokens


100%|██████████| 50/50 [00:00<00:00, 291.21it/s]


Ministral text-only tensor / embedding analysis:
    sample_id          label  \
0           0    declarative   
1           1    declarative   
2           2     imperative   
3           3  interrogative   
4           4     imperative   
5           5    declarative   
6           6     imperative   
7           7  interrogative   
8           8     imperative   
9           9    exclamatory   
10         10     imperative   
11         11     imperative   
12         12    exclamatory   
13         13     imperative   
14         14    declarative   
15         15     imperative   
16         16    declarative   
17         17    declarative   
18         18     imperative   
19         19     imperative   
20         20    declarative   
21         21     imperative   
22         22     imperative   
23         23     imperative   
24         24    declarative   
25         25     imperative   
26         26    declarative   
27         27     imperative   
28         28    decla

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# Voxtral speech+text tensor / embedding analysis

import os
import torch
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files

file_path = "test_dataset_final.csv"
audio_dir = "audio_files"

text_col = "transcript"
audio_col = "file_name"
label_col = "label"

df = pd.read_csv(file_path)
df = df.dropna(subset=[text_col, audio_col, label_col]).reset_index(drop=True)

N_ANALYSIS = 50
small_df = df.head(N_ANALYSIS).copy()

print("Running Voxtral speech+text tensor / embedding analysis")
print("Number of samples:", len(small_df))

if "model" not in globals() or "processor" not in globals():
    raise RuntimeError("model and processor are not loaded. Run your Voxtral model loading cell first.")

# Fix pad token if needed
if hasattr(processor, "tokenizer"):
    tok = processor.tokenizer

    if tok.pad_token is None or tok.pad_token_id is None or tok.pad_token_id < 0:
        tok.pad_token = tok.eos_token
        tok.pad_token_id = tok.eos_token_id

    tok.padding_side = "left"

    try:
        model.config.pad_token_id = tok.pad_token_id
        model.generation_config.pad_token_id = tok.pad_token_id
    except Exception:
        pass

def make_voxtral_text_prompt(transcript):
    prompt = f"""You are a sentence type classifier.

Classify the sentence into exactly one of these labels:

declarative: a normal statement or information-giving sentence.
interrogative: a question or sentence asking for information.
imperative: a command, request, advice, instruction, or order.
exclamatory: a sentence showing strong emotion, surprise, excitement, or emphasis.

Transcript:
{transcript}

Return only one label from:
declarative, exclamatory, imperative, interrogative

Answer with only the label.

Label:"""

    return prompt

def make_voxtral_speech_text_prompt(transcript):
    prompt = f"""You are a sentence type classifier.

You will receive both the speech audio and its transcript.

Classify the sentence into exactly one of these labels:

declarative: a normal statement or information-giving sentence.
interrogative: a question or sentence asking for information.
imperative: a command, request, advice, instruction, or order.
exclamatory: a sentence showing strong emotion, surprise, excitement, or emphasis.

Use both the audio and the transcript. The transcript gives the words, and the audio may give tone, emphasis, and intonation.

Transcript:
{transcript}

Return only one label from:
declarative, exclamatory, imperative, interrogative

Answer with only the label.

Label:"""

    return prompt

def build_voxtral_text_only_inputs(transcript):
    prompt = make_voxtral_text_prompt(transcript)

    conversation = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                }
            ],
        }
    ]

    try:
        inputs = processor.apply_chat_template(
            conversation,
            tokenize=True,
            return_tensors="pt",
            return_dict=True
        )
    except TypeError:
        inputs = processor.apply_chat_template(
            conversation,
            return_tensors="pt",
            return_dict=True
        )

    return inputs, conversation, prompt

def build_voxtral_speech_text_inputs(audio_path, transcript):
    prompt = make_voxtral_speech_text_prompt(transcript)

    conversation = [
        {
            "role": "user",
            "content": [
                {
                    "type": "audio",
                    "path": audio_path
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ],
        }
    ]

    try:
        inputs = processor.apply_chat_template(
            conversation,
            tokenize=True,
            return_tensors="pt",
            return_dict=True
        )
    except TypeError:
        inputs = processor.apply_chat_template(
            conversation,
            return_tensors="pt",
            return_dict=True
        )

    return inputs, conversation, prompt

def find_text_embedding_layer(model):
    candidates = []

    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Embedding):
            candidates.append((name, module))

    print("\nFound embedding layers:")

    for name, module in candidates:
        print(name, "weight shape:", tuple(module.weight.shape), "device:", module.weight.device)

    if len(candidates) == 0:
        raise ValueError("No torch.nn.Embedding layer found inside the model.")

    preferred_words = [
        "embed_tokens",
        "tok_embeddings",
        "word_embeddings",
        "wte"
    ]

    for word in preferred_words:
        for name, module in candidates:
            if word in name.lower():
                print("\nUsing text embedding layer:", name)
                return module

    candidates = sorted(
        candidates,
        key=lambda x: x[1].weight.shape[0],
        reverse=True
    )

    print("\nUsing fallback text embedding layer:", candidates[0][0])
    return candidates[0][1]

text_embedding_layer = find_text_embedding_layer(model)

def get_text_mean_embedding(inputs):
    emb_layer = text_embedding_layer
    emb_device = emb_layer.weight.device

    input_ids = inputs["input_ids"].to(emb_device)

    with torch.no_grad():
        text_emb = emb_layer(input_ids)

    text_emb = text_emb.detach().float().cpu()

    # shape: batch, token_length, hidden_dim
    mean_emb = text_emb.mean(dim=1).squeeze(0)

    return mean_emb.numpy(), tuple(text_emb.shape)

def get_audio_mean_feature(inputs):
    audio_keys = [
        k for k in inputs.keys()
        if "audio" in k.lower() or "feature" in k.lower() or "input_values" in k.lower()
    ]

    possible_audio_keys = [
        "input_features",
        "audio_features",
        "input_values",
        "audio_values"
    ]

    audio_key = None

    for k in possible_audio_keys:
        if k in inputs and torch.is_tensor(inputs[k]):
            audio_key = k
            break

    if audio_key is None:
        return None, None, audio_keys, None

    feats = inputs[audio_key].detach().float().cpu()
    raw_shape = tuple(feats.shape)

    x = feats.squeeze(0)

    if x.ndim == 2:
        # x can be frames x feature_dim or feature_dim x frames
        if x.shape[-1] in [80, 128, 256, 512]:
            vec = x.mean(dim=0)
        elif x.shape[0] in [80, 128, 256, 512]:
            vec = x.mean(dim=1)
        else:
            if x.shape[0] > x.shape[1]:
                vec = x.mean(dim=0)
            else:
                vec = x.mean(dim=1)

    elif x.ndim == 3:
        vec = x.reshape(-1, x.shape[-1]).mean(dim=0)

    else:
        flat = x.reshape(-1)
        vec = torch.tensor([
            flat.mean(),
            flat.std(),
            flat.min(),
            flat.max()
        ])

    return vec.numpy(), raw_shape, audio_keys, audio_key

analysis_rows = []

voxtral_text_only_vecs = []
voxtral_speech_text_text_vecs = []
voxtral_speech_text_audio_vecs = []

for i, row in tqdm(small_df.iterrows(), total=len(small_df)):
    audio_path = os.path.join(audio_dir, str(row[audio_col]))
    transcript = str(row[text_col])
    label = str(row[label_col])

    text_inputs, text_conv, text_prompt = build_voxtral_text_only_inputs(transcript)

    speech_text_inputs, speech_text_conv, speech_text_prompt = build_voxtral_speech_text_inputs(
        audio_path,
        transcript
    )

    if i == 0:
        print("\nVoxtral text-only processor keys:")
        print(list(text_inputs.keys()))

        print("\nVoxtral speech+text processor keys:")
        print(list(speech_text_inputs.keys()))

        print("\nVoxtral speech+text conversation content types:")
        for msg in speech_text_conv:
            print("Role:", msg["role"])
            for item in msg["content"]:
                print("  type:", item["type"])

    text_vec, text_shape = get_text_mean_embedding(text_inputs)
    speech_text_text_vec, speech_text_text_shape = get_text_mean_embedding(speech_text_inputs)

    speech_text_audio_vec, speech_text_audio_shape, speech_text_audio_keys, audio_key_used = get_audio_mean_feature(
        speech_text_inputs
    )

    voxtral_text_only_vecs.append(text_vec)
    voxtral_speech_text_text_vecs.append(speech_text_text_vec)

    if speech_text_audio_vec is not None:
        voxtral_speech_text_audio_vecs.append(speech_text_audio_vec)

    text_cos = cosine_similarity(
        text_vec.reshape(1, -1),
        speech_text_text_vec.reshape(1, -1)
    )[0][0]

    analysis_rows.append({
        "sample_id": i,
        "file_name": row[audio_col],
        "label": label,

        "voxtral_text_only_input_ids_len": int(text_inputs["input_ids"].shape[-1]),
        "voxtral_speech_text_input_ids_len": int(speech_text_inputs["input_ids"].shape[-1]),

        "voxtral_text_only_text_embedding_shape": str(text_shape),
        "voxtral_speech_text_text_embedding_shape": str(speech_text_text_shape),

        "voxtral_speech_text_audio_feature_shape": str(speech_text_audio_shape),
        "voxtral_speech_text_audio_keys": str(speech_text_audio_keys),
        "voxtral_audio_key_used": str(audio_key_used),

        "text_embedding_cosine_text_only_vs_speech_text": float(text_cos),

        "text_prompt_char_length": len(text_prompt),
        "speech_text_prompt_char_length": len(speech_text_prompt)
    })

analysis_df = pd.DataFrame(analysis_rows)

print("\nVoxtral tensor / embedding analysis:")
print(analysis_df)

print("\nSummary:")

print("Average Voxtral text-only input_ids length:")
print(analysis_df["voxtral_text_only_input_ids_len"].mean())

print("\nAverage Voxtral speech+text input_ids length:")
print(analysis_df["voxtral_speech_text_input_ids_len"].mean())

print("\nAverage text embedding cosine text-only vs speech+text inside Voxtral:")
print(analysis_df["text_embedding_cosine_text_only_vs_speech_text"].mean())

print("\nAudio key used:")
print(analysis_df["voxtral_audio_key_used"].value_counts())

analysis_df.to_csv("voxtral_speech_text_embedding_analysis_n50.csv", index=False)

np.save(
    "voxtral_text_only_text_mean_embeddings_n50.npy",
    np.stack(voxtral_text_only_vecs)
)

np.save(
    "voxtral_speech_text_text_mean_embeddings_n50.npy",
    np.stack(voxtral_speech_text_text_vecs)
)

if len(voxtral_speech_text_audio_vecs) > 0:
    np.save(
        "voxtral_speech_text_audio_mean_features_n50.npy",
        np.stack(voxtral_speech_text_audio_vecs)
    )

print("\nSaved files:")
print("voxtral_speech_text_embedding_analysis_n50.csv")
print("voxtral_text_only_text_mean_embeddings_n50.npy")
print("voxtral_speech_text_text_mean_embeddings_n50.npy")
print("voxtral_speech_text_audio_mean_features_n50.npy")

files.download("voxtral_speech_text_embedding_analysis_n50.csv")

Running Voxtral speech+text tensor / embedding analysis
Number of samples: 50

Found embedding layers:
model.language_model.embed_tokens weight shape: (131072, 3072) device: cuda:0

Using text embedding layer: model.language_model.embed_tokens


  8%|▊         | 4/50 [00:00<00:03, 14.72it/s]


Voxtral text-only processor keys:
['input_ids', 'attention_mask']

Voxtral speech+text processor keys:
['input_ids', 'attention_mask', 'input_features']

Voxtral speech+text conversation content types:
Role: user
  type: audio
  type: text


100%|██████████| 50/50 [00:02<00:00, 21.08it/s]



Voxtral tensor / embedding analysis:
    sample_id      file_name          label  voxtral_text_only_input_ids_len  \
0           0    0_0_d45.wav    declarative                              121   
1           1    0_0_d47.wav    declarative                              118   
2           2   0_0_d215.wav     imperative                              116   
3           3    0_0_d49.wav  interrogative                              118   
4           4   0_0_d159.wav     imperative                              125   
5           5    0_0_d50.wav    declarative                              113   
6           6   0_0_d403.wav     imperative                              120   
7           7    0_0_d63.wav  interrogative                              120   
8           8   0_0_d681.wav     imperative                              126   
9           9   0_0_d684.wav    exclamatory                              120   
10         10   0_0_d704.wav     imperative                              117   
11

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>